## A Notebook to make some quick data visualizations

### Toy data

In [ ]:
# Imports
import torch

import numpy as np
import matplotlib.pyplot as plt

from point_clouds import PointCloudGenerator, TargetDistribution
from sklearn.datasets import make_checkerboard

In [ ]:
# Instatiate generator object
data_generator = PointCloudGenerator(num_samples=500, target_name="checkerboard")

In [ ]:
# Draw samples x0~p0 and x1~p1
x0 = data_generator.draw_source()
x1 = data_generator.draw_target()
print(f"x0: {x0.shape}; x1:{x1.shape}")

In [ ]:
# Visualize the data
fig, (ax0, ax1) = plt.subplots(nrows=1, ncols=2, sharex=1, sharey=1, figsize=(10, 5))
ax0.scatter(x0[:,0], x0[:,1], s=4, alpha=0.5); ax0.set_title("Stadard gaussian"); ax0.grid()
ax1.scatter(x1[:,0], x1[:,1], s=4, alpha=0.5); ax1.set_title("Two moons"); ax1.grid()
plt.tight_layout()
plt.show()

### ChEMBL (SDF) data

See the RDKit docs at https://www.rdkit.org/docs/GettingStartedInPython.html

In [3]:
import matplotlib.pyplot as plt

from tqdm import tqdm
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, Descriptors
from pprint import pprint
import py3Dmol

In [4]:
def MolTo3DView(mol, size=(300, 300), style="stick", surface=False, opacity=0.5):
    """Draw molecule in 3D
    
    Args:
    ----
        mol: rdMol, molecule to show
        size: tuple(int, int), canvas size
        style: str, type of drawing molecule
               style can be 'line', 'stick', 'sphere', 'carton'
        surface, bool, display SAS
        opacity, float, opacity of surface, range 0.0-1.0
    Return:
    ----
        viewer: py3Dmol.view, a class for constructing embedded 3Dmol.js views in ipython notebooks.
    """
    assert style in ('line', 'stick', 'sphere', 'carton')
    mblock = Chem.MolToMolBlock(mol)
    viewer = py3Dmol.view(width=size[0], height=size[1])
    viewer.addModel(mblock, 'mol')
    viewer.setStyle({style:{}})
    if surface:
        viewer.addSurface(py3Dmol.SAS, {'opacity': opacity})
    viewer.zoomTo()
    return viewer

In [5]:
# SDF path
sdf_path = Path("/home/vhucke/Data/chembl/chembl_37.sdf")

In [6]:
def mol2conf(mol: Chem.rdchem.Mol):
    '''Convert rdkit.Mol with 2D coord. to rdkit.Mol with 3D coord.'''
    if mol is not None:
        mol = Chem.AddHs(mol)
        params = AllChem.ETKDGv3() # type: ignore
        params.randomSeed = 0xf00d # optional random seed for reproducibility
        conformer_idx = AllChem.EmbedMolecule(mol, params) # type: ignore
        if conformer_idx == -1:
            print(f"Warning: Embedding failed with code {conformer_idx}.")
        return mol
    else:
        return None

def smi2conf(smiles: str):
    '''Convert SMILES to rdkit.Mol with 3D coord.'''
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol) # type: ignore
        AllChem.MMFFOptimizeMolecule(mol, maxIters=200) # type: ignore
        return mol
    else:
        return None

smi = 'C1=CC=C2C(=C1)C(=CN2)C[C@@H](C(=O)O)N'
conf = smi2conf(smi)
viewer = MolTo3DView(conf, size=(600, 300), style='stick')
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
# Open SDF
with Chem.SDMolSupplier(str(sdf_path)) as suppl:    # NOTE: No 3D structures, at least no information about it!
    mol = suppl[10000]      # Get just one element from the SDF (which is a list of many MOL files)
    conf = mol2conf(mol)
    viewer = MolTo3DView(conf, size=(600, 300), style='stick')
    viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
Descriptors.MolLogP(mol), Descriptors.MolWt(mol) # type: ignore # TODO: Check how many embeddings do not work!

(0.4304000000000013, 580.5830000000004)